# CCGA symbolic notebook (SymPy)

`kingdon` multivectors accept **SymPy** coefficients directly, so the same
`Algebra(5,3)` works for exact symbolic computation (see
`ccga/point.py::verify_point_properties`).

Note: `print_null` / `to_null_basis` are **numeric-only** (they cast
coefficients to `float`). For symbolic multivectors use the
`null_coeffs_sym` reader defined below.

Select the **CCGA (.venv)** kernel.

In [1]:
import sympy as sp
from ccga.algebra import (alg, e1, e2, eo, einf, eo1, eo2, eo3,
                          einf1, einf2, einf3, Iod, Iinfd, Iinf)

# Reciprocal frame for the 8 null basis vectors (eo_i.einf_i = -1, e_i.e_i = 1)
_recip = {'eo1': -einf1, 'eo2': -einf2, 'eo3': -einf3, 'e1': e1, 'e2': e2,
          'einf1': -eo1, 'einf2': -eo2, 'einf3': -eo3}

def null_coeffs_sym(mv):
    """Symbolic null-basis coefficients of a GRADE-1 multivector (dict, simplified)."""
    out = {}
    for name, r in _recip.items():
        c = sp.nsimplify(sp.simplify((mv | r).e))
        if c != 0:
            out[name] = c
    return out

def scalar(mv):
    """Simplified scalar (grade-0) part (floats -> rationals)."""
    return sp.nsimplify(sp.simplify(mv.e))

## 1. The symbolic point and its §2 properties

In [2]:
x, y = sp.symbols('x y', real=True)

def sym_point(cx, cy):
    return (eo + cx*e1 + cy*e2
            + (cx**2/2)*einf1 + (cy**2/2)*einf2 + cx*cy*einf3)

p = sym_point(x, y)
print('null coeffs of p(x,y):', null_coeffs_sym(p))
print('p^2          =', scalar(p * p))        # 0  (null)
print('p . einf     =', scalar(p | einf))     # -1 (normalization)

null coeffs of p(x,y): {'eo1': 1, 'eo2': 1, 'e1': x, 'e2': y, 'einf1': x**2/2, 'einf2': y**2/2, 'einf3': x*y}
p^2          = 0
p . einf     = -1


In [3]:
# Distance identity:  p.q = -1/2 [ (x-x')^2 + (y-y')^2 ]
xp, yp = sp.symbols("x' y'", real=True)
q = sym_point(xp, yp)
lhs = scalar(p | q)
rhs = sp.Rational(-1, 2) * ((x - xp)**2 + (y - yp)**2)
print('p.q       =', lhs)
print('p.q - rhs =', sp.simplify(lhs - rhs))   # 0

p.q       = -x**2/2 + x*x' - x'**2/2 - y**2/2 + y*y' - y'**2/2
p.q - rhs = 0


## 2. §3 result 1 — IPNS grade-1 vector is a general conic

For a symbolic vector $s$, the locus $q\cdot s = 0$ should be
$Ax^2+By^2+Cxy+Dx+Ey+F=0$ with the §3 coefficient map.

In [4]:
so1, so2, so3, se1, se2, si1, si2 = sp.symbols('s_o1 s_o2 s_o3 s_e1 s_e2 s_inf1 s_inf2')
s = (so1*eo1 + so2*eo2 + so3*eo3 + se1*e1 + se2*e2 + si1*einf1 + si2*einf2)

expr = sp.expand(sp.nsimplify((sym_point(x, y) | s).e))
poly = sp.Poly(expr, x, y)
A = poly.coeff_monomial(x**2); B = poly.coeff_monomial(y**2); C = poly.coeff_monomial(x*y)
D = poly.coeff_monomial(x);    E = poly.coeff_monomial(y);    F = poly.coeff_monomial(1)
print('q . s  =', expr)
print()
for nm, val, expect in [('A', A, -so1/2), ('B', B, -so2/2), ('C', C, -so3),
                        ('D', D, se1), ('E', E, se2), ('F', F, -(si1 + si2))]:
    print(f'{nm} = {val}   (expected {expect}, match: {sp.simplify(val-expect)==0})')

q . s  = s_e1*x + s_e2*y - s_inf1 - s_inf2 - s_o1*x**2/2 - s_o2*y**2/2 - s_o3*x*y

A = -s_o1/2   (expected -s_o1/2, match: True)
B = -s_o2/2   (expected -s_o2/2, match: True)
C = -s_o3   (expected -s_o3, match: True)
D = s_e1   (expected s_e1, match: True)
E = s_e2   (expected s_e2, match: True)
F = -s_inf1 - s_inf2   (expected -s_inf1 - s_inf2, match: True)


## 3. Symbolic CGA round point — the isotropic collapse (§3.3 / §3.9)

$p \wedge I_\infty^{\triangleright}$ collapses $\tfrac{x^2}{2}e_{\infty_1}+\tfrac{y^2}{2}e_{\infty_2}+xy\,e_{\infty_3}$
into the single isotropic term $\tfrac{x^2+y^2}{2}$.  The coefficient on
$e_{\infty_1}\!\wedge e_{\infty_2}\!\wedge e_{\infty_3}$ is read off by pairing with its
origin dual $I_o = e_{o_1}\!\wedge e_{o_2}\!\wedge e_{o_3}$.

In [5]:
R = sym_point(x, y) ^ Iinfd            # grade-3 CGA round point
Io = eo1 ^ eo2 ^ eo3
iso = sp.nsimplify(sp.simplify((R | Io).e))
print('isotropic (einf1^einf2^einf3) coefficient =', iso)
print('expected  -(x^2 + y^2)/2                   =', -(x**2 + y**2)/2)
print('match:', sp.simplify(iso + (x**2 + y**2)/2) == 0)

isotropic (einf1^einf2^einf3) coefficient = -x**2/2 - y**2/2
expected  -(x^2 + y^2)/2                   = -x**2/2 - y**2/2
match: True


## 4. CCGA point pair  $p_1 \wedge p_2$  (grade 2)

The CCGA dipole `make_point_pair(p1, p2) = p1 ^ p2` is a bivector (grade 2). We
read its null-bivector components symbolically (extending the grade-1 reader with
reciprocal **bivectors**), then show the reality identity.

For null points ($p_1^2 = p_2^2 = 0$) the geometric square factorizes:
$$(p_1 \wedge p_2)^2 = (p_1\cdot p_2)^2 = \tfrac14\big[(x_1-x_2)^2+(y_1-y_2)^2\big]^2 \ge 0,$$
so a CCGA point pair built from distinct real points is **always real**
($P^2 > 0$), matching `classify`'s reality test.

In [6]:
import itertools
_names2 = ['eo1', 'eo2', 'eo3', 'e1', 'e2', 'einf1', 'einf2', 'einf3']

# Symbolic null-BIVECTOR coefficients of a grade-2 multivector.
# Coefficient of  n_i ^ n_j  is  mv . (recip_j ^ recip_i)  via the reciprocal
# frame (matches the numeric to_null_basis).
def null_coeffs_sym2(mv):
    out = {}
    for i, j in itertools.combinations(range(8), 2):
        ni, nj = _names2[i], _names2[j]
        c = sp.nsimplify(sp.simplify((mv | (_recip[nj] ^ _recip[ni])).e))
        if c != 0:
            out[f'{ni}^{nj}'] = c
    return out

In [7]:
x1, y1, x2, y2 = sp.symbols('x1 y1 x2 y2', real=True)
p1, p2 = sym_point(x1, y1), sym_point(x2, y2)
PP = p1 ^ p2                       # CCGA point pair (dipole)

print('grades present:', sorted({bin(k).count('1') for k in PP.keys()}))   # [2]
print('null-bivector components of p1 ^ p2:')
for blade, coeff in null_coeffs_sym2(PP).items():
    print(f'  {blade:12s} {coeff}')

grades present: [2]
null-bivector components of p1 ^ p2:


  eo1^e1       -x1 + x2
  eo1^e2       -y1 + y2
  eo1^einf1    -x1**2/2 + x2**2/2
  eo1^einf2    -y1**2/2 + y2**2/2
  eo1^einf3    -x1*y1 + x2*y2
  eo2^e1       -x1 + x2
  eo2^e2       -y1 + y2
  eo2^einf1    -x1**2/2 + x2**2/2
  eo2^einf2    -y1**2/2 + y2**2/2
  eo2^einf3    -x1*y1 + x2*y2
  e1^e2        x1*y2 - x2*y1
  e1^einf1     x1*x2*(-x1 + x2)/2
  e1^einf2     x1*y2**2/2 - x2*y1**2/2
  e1^einf3     x1*x2*(-y1 + y2)
  e2^einf1     -x1**2*y2/2 + x2**2*y1/2
  e2^einf2     y1*y2*(-y1 + y2)/2
  e2^einf3     y1*y2*(-x1 + x2)
  einf1^einf2  x1**2*y2**2/4 - x2**2*y1**2/4
  einf1^einf3  x1*x2*(x1*y2 - x2*y1)/2
  einf2^einf3  y1*y2*(-x1*y2 + x2*y1)/2


In [8]:
# The square factorizes (p1, p2 are null: p1^2 = p2^2 = 0):
#   (p1 ^ p2)^2 = (p1 . p2)^2 = dist^4 / 4
sq    = scalar(PP * PP)
dot   = scalar(p1 | p2)
dist2 = (x1 - x2)**2 + (y1 - y2)**2
print('(p1^p2)^2  =', sp.factor(sq))
print('(p1.p2)^2  =', sp.factor(dot**2))
print('equal?     ', sp.simplify(sq - dot**2) == 0)
print('p1.p2      =', sp.factor(dot), '   = -dist^2/2 ?', sp.simplify(dot + dist2/2) == 0)
print('(p1^p2)^2 = dist^4/4 ?', sp.simplify(sq - dist2**2/4) == 0)
print('=> (p1^p2)^2 >= 0 for distinct real points: a CCGA point pair is always real')

(p1^p2)^2  = (x1**2 - 2*x1*x2 + x2**2 + y1**2 - 2*y1*y2 + y2**2)**2/4
(p1.p2)^2  = (x1**2 - 2*x1*x2 + x2**2 + y1**2 - 2*y1*y2 + y2**2)**2/4
equal?      True
p1.p2      = -(x1**2 - 2*x1*x2 + x2**2 + y1**2 - 2*y1*y2 + y2**2)/2    = -dist^2/2 ? True
(p1^p2)^2 = dist^4/4 ? True
=> (p1^p2)^2 >= 0 for distinct real points: a CCGA point pair is always real


## 5. The dipole in center / direction / radius form

Place the two endpoints at $C \pm r\,v$ with **unit** direction $v=(v_x,v_y)$ and
radius $r$. The point map $f(t)=p(C + t\,v)$ is quadratic, so
$$f(t) = C + t\,W + t^2\,v_\infty,\qquad W = \mathrm dC[v]\ (\text{tangent}),\quad v_\infty = \tfrac12 f''\ (\text{ideal point of }v).$$
With $p_1=f(r)$, $p_2=f(-r)$ the dipole collapses to

$$\boxed{\,p_1\wedge p_2 \;=\; -2r\,\big(C + r^2\,v_\infty\big)\wedge W\,}$$

The overall $-2r$ is scale/orientation; the geometry lives in
$(C + r^2 v_\infty)\wedge W$, where $r^2$ is the **signed squared radius** —
$r^2>0$ gives a real pair, $r^2<0$ an **imaginary** pair (no real endpoints). The
square is $P^2 = 4r^4$ for a unit direction (from $P^2 = \mathrm{dist}^4/4$,
$\mathrm{dist}=2r$).

In [9]:
px, py, vx, vy, r = sp.symbols('px py vx vy r', real=True)
p1 = sym_point(px + r*vx, py + r*vy)        # center + r*v
p2 = sym_point(px - r*vx, py - r*vy)        # center - r*v
PP = p1 ^ p2

# pieces = the 0th/1st/2nd derivatives of the point map along v:
C     = sym_point(px, py)                                                   # center point
W     = vx*e1 + vy*e2 + px*vx*einf1 + py*vy*einf2 + (px*vy + py*vx)*einf3    # tangent dC[v]
v_inf = (vx**2/2)*einf1 + (vy**2/2)*einf2 + vx*vy*einf3                      # ideal point of v

print('center  C   :', null_coeffs_sym(C))
print('tangent W   :', null_coeffs_sym(W))
print('dir ideal v_inf :', null_coeffs_sym(v_inf))

closed = -2*r * ((C + r**2 * v_inf) ^ W)
print('\nPP == -2*r*(C + r^2*v_inf) ^ W  :',
      all(sp.simplify(sp.expand(c)) == 0 for c in (PP - closed).values()))

# square: P^2 = dist^4/4 = 4 r^4 (vx^2+vy^2)^2  ->  4 r^4 for a unit direction
print('P^2         :', sp.factor(scalar(PP * PP)), '  -> 4*r^4 for unit v')

center  C   : {'eo1': 1, 'eo2': 1, 'e1': px, 'e2': py, 'einf1': px**2/2, 'einf2': py**2/2, 'einf3': px*py}
tangent W   : {'e1': vx, 'e2': vy, 'einf1': px*vx, 'einf2': py*vy, 'einf3': px*vy + py*vx}
dir ideal v_inf : {'einf1': vx**2/2, 'einf2': vy**2/2, 'einf3': vx*vy}



PP == -2*r*(C + r^2*v_inf) ^ W  : True


P^2         : 4*r**4*(vx**2 + vy**2)**2   -> 4*r^4 for unit v


## 6. CGA(2) reference decomposition (cross-check)

For reference, the standard CGA(2) $=\mathbb R^{3,1}$ dipole, with center
$p=(p_x,p_y,p_w)$, **unit** direction $v=(v_x,v_y,0)$ and radius $r$:

$$
\begin{aligned}
PP &= v_x\,e_{01} + v_y\,e_{02} + (p_xv_y-p_yv_x)\,e_{12}\\
   &\quad + (p\cdot v)\,(p_x\,e_{1\infty}+p_y\,e_{2\infty}+p_w\,e_{0\infty})
          - \tfrac{p_x^2+p_y^2\pm r^2}{2}\,(v_x\,e_{1\infty}+v_y\,e_{2\infty}).
\end{aligned}
$$

The cell below verifies (with $p_w=1$, unit $v$) that this equals $p_1\wedge p_2$
up to the overall $-2r$ normalization, with the **$+r^2$** branch the real pair
and **$-r^2$** the imaginary one. It is the single-$\infty$ shadow of the CCGA
result in §5, $-2r\,(C+r^2 v_\infty)\wedge W$: same $-2r$ scale and the same
center$\wedge$direction ($W$) + radius ($r^2 v_\infty$) split, but CCGA spreads
$e_o,e_\infty$ across $e_{o_1},e_{o_2}$ and $e_{\infty_1},e_{\infty_2},e_{\infty_3}$.

In [10]:
# CGA(2) = R^{3,1} reference cross-check (its own algebra, independent of CCGA above)
from kingdon import Algebra as _Alg
_ga = _Alg(3, 1)
_b = lambda k: _ga.multivector({k: 1})
E1, E2, _ep, _em = _b('e1'), _b('e2'), _b('e3'), _b('e4')
O   = _ep + _em             # CGA origin  (null)
Inf = (_em - _ep) / 2       # CGA infinity (null),  O . Inf = -1

def cga_point(X, Y):
    return O + X*E1 + Y*E2 + (X**2 + Y**2)/2 * Inf

cx, cy, th, rr = sp.symbols('cx cy theta r', real=True)
ux, uy = sp.cos(th), sp.sin(th)                       # unit direction
Dcga = cga_point(cx + rr*ux, cy + rr*uy) ^ cga_point(cx - rr*ux, cy - rr*uy)

pv = cx*ux + cy*uy                                    # p . v   (p_w = 1)
ref = (ux*(O^E1) + uy*(O^E2)
       + (cx*uy - cy*ux)*(E1^E2)
       + pv*(cx*(E1^Inf) + cy*(E2^Inf) + (O^Inf))
       - (cx**2 + cy**2 + rr**2)/2 * (ux*(E1^Inf) + uy*(E2^Inf)))

diff = Dcga - (-2*rr)*ref
print('CGA  p1^p2 == -2r * [reference, +r^2]  (unit v, p_w=1):',
      all(sp.simplify(sp.expand(sp.trigsimp(c))) == 0 for c in diff.values()))
print('(the -r^2 branch is the imaginary-radius dipole)')

CGA  p1^p2 == -2r * [reference, +r^2]  (unit v, p_w=1): True
(the -r^2 branch is the imaginary-radius dipole)


## 7. The triplet on a circle — center / radius / direction form

Three points on a circle of **circumcenter** $C_0$, **radius** $R_c$ and unit
directions $d_k=(\cos\theta_k,\sin\theta_k)$. Since $p(C_0 + R_c\,d_k)$ is
quadratic in the displacement,
$$p_k = C_0 + a_k,\qquad a_k = R_c\,W_k + R_c^{2}\,Q_k,$$
with $W_k=\mathrm dC_0[d_k]$ the tangent and $Q_k$ the ideal point of $d_k$.
Because $C_0$ appears in all three factors and $C_0\wedge C_0=0$, the wedge of
the three points collapses to

$$\boxed{\;T = p_1\wedge p_2\wedge p_3
   = C_0 \wedge\big(a_1\wedge a_2 + a_2\wedge a_3 + a_3\wedge a_1\big)
   \; + \; a_1\wedge a_2\wedge a_3\;}$$

— the circumcenter $C_0$ explicit, with a grade-2 part $B=\sum_{i<j}a_i\wedge a_j$
and a grade-3 part $V=a_1\wedge a_2\wedge a_3$. This is the grade-3 analogue of the
point pair's $-2r\,(C+r^2 v_\infty)\wedge W$ (§5).

In [11]:
import math
cx, cy, Rc = sp.symbols('cx cy R_c', real=True)   # circumcenter (cx,cy), circumradius R_c
angs = [0.4, 2.3, 4.1]                             # three fixed direction angles

def _W(c, s): return c*e1 + s*e2 + cx*c*einf1 + cy*s*einf2 + (cx*s + cy*c)*einf3
def _Q(c, s): return (c**2/2)*einf1 + (s**2/2)*einf2 + c*s*einf3
def _pt(th):
    c, s = math.cos(th), math.sin(th)
    X, Y = cx + Rc*c, cy + Rc*s
    return eo + X*e1 + Y*e2 + (X**2/2)*einf1 + (Y**2/2)*einf2 + X*Y*einf3

T  = _pt(angs[0]) ^ _pt(angs[1]) ^ _pt(angs[2])    # triplet, symbolic center/radius
C0 = eo + cx*e1 + cy*e2 + (cx**2/2)*einf1 + (cy**2/2)*einf2 + cx*cy*einf3
a  = [Rc*_W(math.cos(t), math.sin(t)) + Rc**2*_Q(math.cos(t), math.sin(t)) for t in angs]
B  = (a[0] ^ a[1]) + (a[1] ^ a[2]) + (a[2] ^ a[0])   # NB: parenthesise wedges (^ binds looser than +)
V  =  a[0] ^ a[1] ^ a[2]
form = (C0 ^ B) + V

# verify  T == C0^B + V  (symbolic cx,cy,R_c; chop the float-0.5 basis noise at a test point)
sub = {cx: sp.Rational(7, 5), cy: sp.Rational(-3, 4), Rc: sp.Rational(11, 7)}
print('T == C0^(a1^a2 + a2^a3 + a3^a1) + a1^a2^a3 :',
      all(abs(complex(sp.N(c.subs(sub)))) < 1e-9 for c in (T - form).values()))
print('grades:  C0^B ->', sorted({bin(k).count('1') for k in (C0 ^ B).keys()}),
      '  V ->', sorted({bin(k).count('1') for k in V.keys()}))

T == C0^(a1^a2 + a2^a3 + a3^a1) + a1^a2^a3 : True


grades:  C0^B -> [3]   V -> [3]


## Your scratch space